# PLM Control Guide
Complete workflow for controlling the DLP670S PLM.

**Important:** Close the LightCrafter EVM GUI before running this notebook – only one process can hold the USB connection.

In [ ]:
import numpy as np
import ctypes
import matplotlib.pyplot as plt
from PLMController import PLMController

## 1. Create Controller Instance
Adjust `MAX_FRAMES`, `N` (width), `M` (height), and the PLM monitor position `(x0, y0)` to match your setup.

In [ ]:
MAX_FRAMES = 64
N = 1358   # PLM width
M = 800    # PLM height
x0 = 2560  # X offset (right of main monitor)
y0 = 0

dll_path = r'..\bin\plmctrl.dll'

plm = PLMController(MAX_FRAMES, N, M, dll_path, x0, y0)

## 2. Start UI (Window)
`windowed=True` is useful for testing – it shows the PLM output in a resizable window.

In [ ]:
plm.set_windowed(True)
plm.start_ui()

## 3. Open USB Connection
Must be called **before** any configuration or display commands.

In [ ]:
res = plm.open()
print(f"Open returned: {res} (0 = success)")

## 4. Configure the PLM
The `configure()` method runs the full setup sequence with proper delays:
- Set source to Parallel RGB (24-bit)
- Set port swap (ABC → ABC)
- Set connection type (HDMI or DisplayPort)
- Set video pattern mode
- Update lookup table

Run this **once per power cycle** of the PLM.

In [ ]:
HDMI = 1
DisplayPort = 2
PLAY_ONCE = 0
CONTINUOUS = 1

plm.configure(play_mode=CONTINUOUS, connection_type=HDMI)
print("Configuration complete")

If `configure()` fails, you can run the steps individually for debugging:

In [ ]:
# Individual steps (same as configure() does internally):
# plm.set_source(0, 1)              # Parallel RGB, 24-bit
# plm.set_port_swap(0, 0)           # Port 1: ABC->ABC
# plm.set_port_swap(1, 0)           # Port 2: ABC->ABC
# plm.set_pixel_mode(HDMI)          # HDMI=1, DisplayPort=2
# import time
# if plm.get_connection_type() != 1:
#     plm.set_connection_type(HDMI)
#     time.sleep(5.5)
# plm.set_video_pattern_mode()
# time.sleep(2.0)
# plm.update_lut(CONTINUOUS, HDMI)

## 5. Set Phase Map & Lookup Table
These define how the PLM maps grayscale values to phase levels.

In [ ]:
# Phase levels (voltage levels for the DMD)
phase_levels = np.array([0.004, 0.017, 0.036, 0.058, 0.085, 0.117, 0.157,
                         0.217, 0.296, 0.4, 0.5, 0.605, 0.713, 0.82,
                         0.922, 0.981, 1.0], dtype=np.float32)
plm.set_lookup_table(phase_levels)

# Phase map (which mirrors/voltages are active for each gray code)
phase_map = np.array([
    [0,0,0,0], [1,0,0,0], [0,1,0,0], [1,1,0,0],
    [0,0,1,0], [1,0,1,0], [0,1,1,0], [1,1,1,0],
    [0,0,0,1], [1,0,0,1], [0,1,0,1], [1,1,0,1],
    [0,0,1,1], [1,0,1,1], [0,1,1,1], [1,1,1,1],
])
phase_map_order = (12, 8, 4, 14, 0, 6, 10, 2, 13, 5, 9, 1, 15, 7, 11, 3)
phase_map = phase_map[phase_map_order, :]
plm.set_phase_map(phase_map)
print("Phase map and LUT set")

## 6. Generate & Bitpack Holograms

### Method A: Bitpack and insert one frame at a time (simplest)

In [ ]:
num_holograms = 24
phase = np.random.rand(num_holograms, M, N).astype(np.float32)

frame = plm.bitpack_holograms_gpu(phase)
plm.insert_frames(frame, 0, format=1)  # format 1 = RGBA
print(f"Inserted frame, shape={frame.shape}")

### Method B: Bitpack and insert all at once (faster)

In [ ]:
num_holograms = 24
frames = np.zeros((MAX_FRAMES, 2*M, 4*2*N), dtype=np.uint8)
phase = np.zeros((num_holograms, M, N), dtype=np.float32)

for i in range(MAX_FRAMES):
    a = np.linspace(0, 0, N)[None, :]
    b = np.linspace(0, i*2 + 1, M)[:, None]
    ph = np.mod(a + b, 1)
    phase[:] = np.tile(ph[None, :, :], (num_holograms, 1, 1))

    phase_ptr = phase.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    frame_ptr = frames[i].ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))
    plm.bitpack_holograms_gpu_ptr(phase_ptr, frame_ptr, num_holograms)

plm.insert_frames(frames, 0, format=1)
print(f"Inserted {MAX_FRAMES} frames")

### Method C: Bitpack and insert inline (minimum copies)

In [ ]:
for i in range(MAX_FRAMES):
    phase = np.random.rand(num_holograms, M, N).astype(np.float32)
    plm.bitpack_and_insert_gpu(phase, i)
print(f"Bitpacked and inserted {MAX_FRAMES} frames inline")

## 7. Set Frame Sequence and Play

The frame sequence defines what order the inserted frames play in.

In [ ]:
# Sequence of frame indices to play (length must equal MAX_FRAMES)
sequence = np.arange(MAX_FRAMES, dtype=np.uint64)
plm.set_frame_sequence(sequence)

In [ ]:
# Start playback
plm.start_sequence(MAX_FRAMES)
print("Playing sequence – call plm.stop() to stop")

### Manual play/stop (alternative to start_sequence)

In [ ]:
# plm.play()
# import time; time.sleep(3)
# plm.stop()

## 8. Cleanup
Always run these cells to properly disconnect and release resources.

In [ ]:
plm.stop()
plm.stop_ui()
plm.lib.Close()
print("Cleanup complete")

## 9. Visualize Phase Data (Debug)
Check what your phase patterns look like before sending to the PLM.

In [ ]:
sample_phase = np.random.rand(24, M, N).astype(np.float32)
frame = plm.bitpack_holograms_gpu(sample_phase)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(sample_phase[0], cmap='gray')
plt.title("Phase pattern 0")
plt.subplot(1, 3, 2)
plt.imshow(frame[:M, :4*N], cmap='gray')
plt.title("Bitpacked frame (top)")
plt.subplot(1, 3, 3)
plt.imshow(frame[M:, :4*N], cmap='gray')
plt.title("Bitpacked frame (bottom)")
plt.tight_layout()
plt.show()

## Quick Reference

| Step | Method | Notes |
|------|--------|-------|
| Init | `PLMController(MAX_FRAMES, N, M, dll_path, x0, y0)` | |
| Window | `set_windowed(True)` then `start_ui()` | Windowed for testing |
| Connect | `open()` | Must come first |
| Config | `configure(play_mode, conn_type)` | Once per power cycle |
| Phase LUT | `set_lookup_table(levels)` | `float32` array |
| Phase map | `set_phase_map(map)` | `int32` 2D array |
| Bitpack | `bitpack_holograms_gpu(phase)` → frame | `float32` → `uint8` |
| Insert | `insert_frames(frame, offset, format)` | `format=1` for RGBA |
| Sequence | `set_frame_sequence(seq)` | `uint64` array |
| Play | `start_sequence(n)` or `play()` | |
| Stop | `stop()` | |
| Cleanup | `stop_ui()` then `lib.Close()` | |